# Classical Music Kaggle Competition

## Load data

In [3]:
import pandas as pd
import sys
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

# locate repo root so the notebook runs from either the repo root or notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "raw"
SUBS = ROOT / "submissions"

In [4]:
# account.csv and zipcodes.csv need special handling:
#  - account.csv isn't UTF-8 (latin-1 works), zip codes kept as strings to preserve leading zeros
#  - concerts*.csv start with a BOM, so use utf-8-sig
account = pd.read_csv(DATA / "account.csv", encoding="latin-1",
                      dtype={"shipping.zip.code": str, "billing.zip.code": str},
                      parse_dates=["first.donated"])
subscriptions = pd.read_csv(DATA / "subscriptions.csv")
tickets = pd.read_csv(DATA / "tickets_all.csv")
concerts = pd.read_csv(DATA / "concerts.csv", encoding="utf-8-sig")
concerts_1415 = pd.read_csv(DATA / "concerts_2014-15.csv", encoding="utf-8-sig")
zipcodes = pd.read_csv(DATA / "zipcodes.csv", dtype={"Zipcode": str})

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

In [5]:
tables = {"account": account, "subscriptions": subscriptions, "tickets": tickets,
          "concerts": concerts, "concerts_1415": concerts_1415, "zipcodes": zipcodes,
          "train": train, "test": test, "sample_submission": sample_submission}
for name, df in tables.items():
    print(f"{name:18s} {df.shape}")

account            (19833, 10)
subscriptions      (28627, 9)
tickets            (2808, 8)
concerts           (103, 6)
concerts_1415      (6, 5)
zipcodes           (42522, 12)
train              (6941, 2)
test               (2975, 1)
sample_submission  (2975, 2)


In [6]:
train["label"].value_counts(normalize=True)

label
0    0.952168
1    0.047832
Name: proportion, dtype: float64

In [7]:
account.head()

,account.id,shipping.zip.code,billing.zip.code,shipping.city,billing.city,relationship,amount.donated.2013,amount.donated.lifetime,no.donations.lifetime,first.donated
0,001i000000LhSrQ,94102,94102,San Francisco,San Francisco,NaN,0.0,0.0,0,NaT
1,001i000000LhyPD,NaN,94303,NaN,Palo Alto,NaN,0.0,500.0,5,1987-07-04
2,001i000000LhyPE,NaN,94597,NaN,Walnut Creek,NaN,0.0,783.0,7,2008-06-02
3,001i000000LhyPF,NaN,94610,NaN,Oakland,NaN,1500.0,28435.0,28,1986-07-04
4,001i000000LhyPG,NaN,94024,NaN,Los Altos,NaN,300.0,4969.0,16,1997-04-07


## EDA: data quality checks

In [8]:
# IDs and train/test overlap
print("dup account ids:", account["account.id"].duplicated().sum())
print("train/test overlap:", len(set(train["account.id"]) & set(test["ID"])))
print("label rate:", train["label"].mean().round(4))

dup account ids: 0
train/test overlap: 0
label rate: 0.0478


In [9]:
# Missingness per table
for name, df in {"account": account, "subscriptions": subscriptions, "tickets": tickets, "zipcodes": zipcodes}.items():
    print(name); print(df.isna().mean().round(3)[lambda s: s > 0].to_string(), "\n")

account
shipping.zip.code    0.987
billing.zip.code     0.149
shipping.city        0.986
billing.city         0.112
relationship         0.967
first.donated        0.721 

subscriptions
section        0.159
price.level    0.123 

tickets
price.level         0.074
marketing.source    0.792
location            0.016
set                 0.008 

zipcodes
Lat                    0.015
Long                   0.015
TaxReturnsFiled        0.321
EstimatedPopulation    0.321
TotalWages             0.322 



In [10]:
# Account: donations (heavy-tailed) and consistency checks
print(account[["amount.donated.2013", "amount.donated.lifetime", "no.donations.lifetime"]].describe().T)
print("2013 > lifetime:", (account["amount.donated.2013"] > account["amount.donated.lifetime"]).sum())
print("first.donated range:", account["first.donated"].min(), "->", account["first.donated"].max())
d = train.merge(account, on="account.id")
print(d.groupby(d["first.donated"].dt.year.fillna(0).astype(int))["label"].agg(["mean", "size"]).tail(6))

                           count         mean           std  min  25%  50%  \
amount.donated.2013      19833.0    85.204938   2224.581861  0.0  0.0  0.0   
amount.donated.lifetime  19833.0  1605.946169  33121.138022  0.0  0.0  0.0   
no.donations.lifetime    19833.0     1.631473      4.974458  0.0  0.0  0.0   

                          75%        max  
amount.donated.2013       0.0   211400.0  
amount.donated.lifetime  25.0  2287650.0  
no.donations.lifetime     1.0       95.0  
2013 > lifetime: 0
first.donated range: 1985-07-04 00:00:00 -> 2014-09-11 00:00:00
                   mean  size
first.donated                
2009           0.421053    19
2010           0.272727    22
2011           0.250000    28
2012           0.300000    40
2013           0.358974    39
2014           0.414634    41


In [11]:
# Zip code formats (billing)
z = account["billing.zip.code"].dropna()
print(z.str.len().value_counts().to_dict())
print("non-standard:", z[~z.str.match(r"^\d{4,5}(-\d{4})?$")].unique()[:10])
clean_zip = account["billing.zip.code"].str.extract(r"^(\d+)")[0].str.zfill(5).str[:5]
print("match rate to zipcodes table after cleaning:", clean_zip.dropna().isin(zipcodes["Zipcode"]).mean().round(3))
print("label rate by billing zip missing:", d.groupby(d["billing.zip.code"].isna())["label"].mean().to_dict())

{5: 16202, 10: 479, 4: 164, 7: 19, 6: 8, 8: 3, 3: 2, 9: 1}
non-standard: <StringArray>
['944041913',   'T8N 3Z9',   'T2L 2L2',   'AL5 9FA',    'A-5010',   'M5S 1X7',
   'V6H 1B6',   'CB1 1DP',   'D-69214',   'GL6 9LJ']
Length: 10, dtype: str
match rate to zipcodes table after cleaning: 0.987
label rate by billing zip missing: {False: 0.05569535312866969, True: 0.0}


In [12]:
# Subscriptions: structural missingness by season
print(subscriptions.groupby("season")[["section", "price.level"]].apply(lambda g: g.isna().mean()).round(2).T)
print(subscriptions["subscription_tier"].value_counts().sort_index().to_dict())
print("acct-season dups:", subscriptions.duplicated(["account.id", "season"]).sum())

season       1993-1994  1994-1995  1995-1996  1996-1997  1997-1998  1998-1999  \
section            0.0        0.0        0.0        0.0        0.0        0.0   
price.level        0.0        0.0        0.0        0.0        0.0        0.0   

season       1999-2000  2000-2001  2001-2002  2002-2003  2003-2004  2004-2005  \
section            0.0        0.0        0.0        1.0        1.0        1.0   
price.level        0.0        0.0        0.0        1.0        1.0        1.0   

season       2005-2006  2006-2007  2007-2008  2008-2009  2009-2010  2010-2011  \
section           0.01       0.01        0.0        0.0        0.0       0.02   
price.level       0.01       0.01        0.0        0.0        0.0       0.00   

season       2011-2012  2012-2013  2013-2014  
section            0.0        0.0        1.0  
price.level        0.0        0.0        0.0  
{0.5: 2776, 1.0: 11331, 1.5: 141, 2.0: 13333, 2.5: 9, 3.0: 484, 3.5: 3, 4.0: 493, 4.5: 1, 5.0: 34, 6.0: 7, 7.0: 1, 8.0: 3, 9.0:

In [13]:
# Tickets: mixed-type price.level, missing set/location, joinability to concerts
print(tickets["price.level"].value_counts().to_dict())
print(tickets[["set", "location"]].isna().sum().to_dict(), "| acct-season-set dups:", tickets.duplicated(["account.id", "season", "set"]).sum())
m = tickets.merge(concerts[["season", "set", "location", "concert.name"]].drop_duplicates(),
                  on=["season", "set", "location"], how="left")
print("ticket->concert match rate:", m["concert.name"].notna().mean().round(4))

{'4': 1042, '3': 563, '2': 489, '1': 435, '0': 35, 'Adult': 14, 'Youth': 14, 'GA': 6, '4.0': 2}
{'set': 22, 'location': 44} | acct-season-set dups: 4
ticket->concert match rate: 0.9922


## Feature pipeline
All feature logic lives in `features.py` so train and test use identical code.

In [14]:
from src.features import load_tables, build_features, GEO_COLS

T = load_tables()
X_train = build_features(train["account.id"], T)
X_test = build_features(test["ID"], T)
y_train = train.set_index("account.id")["label"].loc[X_train.index]

print(X_train.shape, X_test.shape, list(X_train.columns) == list(X_test.columns))

(6941, 105) (2975, 105) True


In [15]:
# Sanity checks: NaNs only in geo columns, strongest univariate correlations with the label
na = X_train.isna().sum(); print(na[na > 0].to_dict())
X_train.corrwith(y_train).dropna().sort_values(key=abs, ascending=False).head(15)

{'geo_lat': 1006, 'geo_lon': 1006, 'geo_log_pop': 1061, 'geo_log_wages': 1259, 'geo_wage_per_return': 1259, 'dist_sf': 1006, 'dist_berkeley': 1006, 'dist_peninsula': 1006, 'dist_walnutcreek': 1006, 'dist_min': 1006}


sub_subscribed_2013       0.824701
sub_n_last3               0.814775
sub_current_streak        0.729259
sub_n_seasons             0.589768
sub_longest_streak        0.584045
don_gave_2013             0.568521
don_2013_log              0.542790
sub_seasons_since_last   -0.524047
don_n                     0.496608
sub_frac_last3            0.478206
don_lifetime_log          0.357089
don_avg_gift_log          0.295865
sub_seats_last            0.294312
sub_tier_last             0.286363
sub_pkg_Trio              0.284977
dtype: float64

In [16]:
X_train.head()

,sub_ever,sub_subscribed_2013,sub_n_seasons,sub_seasons_since_last,sub_years_since_first,sub_current_streak,sub_longest_streak,sub_longest_gap,sub_lapsed_returned,sub_n_last3,sub_frac_last3,sub_price_level_last,sub_price_level_change,sub_price_level_mean,sub_tier_last,sub_seats_last,sub_seats_change,sub_seats_mean,sub_multiple_last,sub_pkg_Full,sub_pkg_Quartet,sub_pkg_Trio,sub_pkg_Other,sub_loc_San Francisco,sub_loc_Berkeley Sunday,sub_loc_Berkeley Saturday,sub_loc_Peninsula,sub_loc_Contra Costa,sub_loc_Santa Rosa,sub_loc_Orange County,sub_loc_Other,sub_lastsec_Orchestra,sub_lastsec_Balcony Front,sub_lastsec_Premium Orchestra,sub_lastsec_Balcony,sub_lastsec_Dress Circle,sub_lastsec_Balcony Rear,sub_lastsec_Orchestra Front,sub_lastsec_Box,sub_lastsec_Other,don_donor,don_gave_2013,don_2013_log,don_lifetime_log,don_n,don_avg_gift_log,don_share_2013,don_years_since_first,rel_present,zip_missing,zip_unmatched,zip_in_ca,geo_lat,geo_lon,geo_log_pop,geo_log_wages,geo_wage_per_return,dist_sf,dist_berkeley,dist_peninsula,dist_walnutcreek,dist_min,tk_has_tickets,tk_n_purchases,tk_n_tickets,tk_n_seasons,tk_seasons_since_last,tk_n_distinct_concerts,tk_avg_price_level,tk_avg_seats,tk_single_share,tk_any_multiple,tk_family_share,tk_special_price,tk_src_known,tk_src_web,tk_matched_share,tk_single_multi_show,tk_loc_San Francisco,tk_loc_Berkeley Saturday,tk_loc_Berkeley Sunday,tk_loc_Peninsula,tk_loc_Other,taste_era_baroque,taste_era_classical,taste_era_romantic,taste_era_modern,taste_is_vocal,taste_is_holiday,taste_is_tentpole,taste_n_soloists,taste_comp_BACH,taste_comp_HANDEL,taste_comp_VIVALDI,taste_comp_MOZART,taste_comp_HAYDN,taste_comp_BEETHOVEN,taste_comp_TELEMANN,taste_comp_CORELLI,taste_sim_composer,taste_sim_era,tk13_n_tickets,tk13_n_shows,tk13_ge3_shows,acct_id_rank
account.id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
001i000000NuRo3,0.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,25.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.921747
001i000000NuRxd,0.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,25.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.935864
001i000000NuQGN,0.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,37.91,-122.07,9.739674,20.306603,61296.428731,34.083614,17.315198,53.874962,0.991843,0.991843,0.0,0.0,0.0,0.0,25.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.629153
001i000000NuPfL,1.0,0.0,1.0,11.0,12.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,-1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,37.68,-122.41,8.377011,19.112493,75627.836047,10.911799,24.714711,34.756805,39.094849,10.911799,0.0,0.0,0.0,0.0,25.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.514345
001i000000NuQkP,0.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,

## Modeling
All preprocessing/selection/resampling is inside the pipeline (`modeling.py`), so it is refit on each training fold. CV = repeated stratified 5-fold, metric = AUROC.

In [17]:
from src.modeling import (build_pipeline, make_cv, compare, high_corr_pairs,
                          permutation_importance_table, write_submission)
import numpy as np

y = y_train.to_numpy()
cv = make_cv(n_splits=5, n_repeats=3)

### 1. Correlation matrix

In [18]:
high_corr_pairs(X_train, threshold=0.9).head(20)

,a,b,abs_corr
0,dist_berkeley,dist_walnutcreek,0.999921
1,dist_sf,dist_berkeley,0.999894
2,dist_sf,dist_min,0.999820
3,dist_walnutcreek,dist_min,0.999813
4,dist_berkeley,dist_min,0.999807
5,dist_sf,dist_walnutcreek,0.999729
6,dist_peninsula,dist_min,0.999713
7,dist_sf,dist_peninsula,0.999248
8,dist_peninsula,dist_walnutcreek,0.999239
9,dist_berkeley,dist_peninsula,0.999136


### 2. Models x imbalance handling (none / class weights / SMOTE)

In [19]:
configs = {}
for model in ["logreg", "rf", "xgb"]:
    for imb in ["none", "weight", "smote"]:
        configs[f"{model} | {imb}"] = build_pipeline(model, imbalance=imb)
results = compare(configs, X_train, y, cv)
results

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 inst

,auroc_mean,auroc_std,n_folds
rf | none,0.981860,0.007683,15.0
rf | weight,0.981483,0.007213,15.0
rf | smote,0.979652,0.008866,15.0
xgb | none,0.979307,0.007252,15.0
logreg | none,0.975412,0.011600,15.0
xgb | weight,0.975298,0.007982,15.0
logreg | weight,0.972563,0.010823,15.0
xgb | smote,0.971640,0.009917,15.0
logreg | smote,0.970660,0.011477,15.0


### 3. Feature selection (correlation filter, then permutation-importance top-k), done inside CV

In [20]:
sel = {"all features": build_pipeline("logreg", "weight", C=0.03),
       "corr<0.9":     build_pipeline("logreg", "weight", C=0.03, corr_threshold=0.9),
       "corr<0.9 + top30": build_pipeline("logreg", "weight", C=0.03, corr_threshold=0.9, top_k=30)}
compare(sel, X_train, y, cv)

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 inst

,auroc_mean,auroc_std,n_folds
all features,0.975633,0.010404,15.0
corr<0.9,0.973355,0.011853,15.0
corr<0.9 + top30,0.959559,0.044132,15.0


In [21]:
# Permutation importance of a fitted pipeline on a held-out split
permutation_importance_table(build_pipeline("rf", "weight", corr_threshold=0.9), X_train, y).head(20)

acct_id_rank              0.033193
geo_wage_per_return       0.004066
sub_seasons_since_last    0.003375
dist_berkeley             0.002402
sub_frac_last3            0.001703
don_lifetime_log          0.000860
sub_subscribed_2013       0.000793
sub_price_level_last      0.000668
taste_era_baroque         0.000657
sub_current_streak        0.000655
don_n                     0.000487
zip_in_ca                 0.000485
sub_n_seasons             0.000475
tk13_n_shows              0.000410
sub_tier_last             0.000350
sub_loc_Other             0.000314
zip_missing               0.000309
geo_lat                   0.000265
don_years_since_first     0.000261
tk_n_tickets              0.000252
dtype: float64

### 4. Write a submission (fit on all training data, positive-class probabilities)

In [22]:
write_submission(build_pipeline("xgb", "weight"), X_train, y, X_test, path=SUBS / "submission_xgb_baseline.csv")

,ID,Predicted
0,001i000000NuQ6Y,0.029738
1,001i000000NuQXz,0.184545
2,001i000000NuRDC,0.060413
3,001i000000NuQeg,0.010275
4,001i000000NuOQc,0.204751
...,...,...
2970,001i000000NuP3y,0.000761
2971,001i000000NuRib,0.002503
2972,001i000000NuP7J,0.000305
2973,001i000000NuRJd,0.080253


## Round 2: feature pruning, tuning, ensemble
Everything below uses the same feature matrix and the same leakage-safe pipelines as above. XGBoost is the evaluator for feature pruning. In the tuning step both boosting models are searched (XGBoost, and scikit-learn's prepackaged HistGradientBoosting), and the final model averages all four tuned models. Selection and tuning use different repeated-CV seeds from the final scoring, so the final numbers are not scored on the folds that picked the settings (they are still the same training rows, so treat them as slightly optimistic).

In [23]:
import itertools
from sklearn.model_selection import RepeatedStratifiedKFold
from src.modeling import (RankAverager, build_ensemble, cv_permutation_importance, fold_scores,
                          grid_search, hgb_random_search, save_config, xgb_random_search)

### 5. Feature pruning
Evaluator = XGBoost (unweighted). (a) drop-one-group ablation, (b) cross-validated permutation importance, (c) accuracy vs. number of top-ranked features.

In [24]:
def group(c):
    if c == "acct_id_rank": return "id"
    if c.startswith(("sub_pkg_", "sub_loc_", "sub_lastsec_")): return "sub_cat"
    if c.startswith("sub_"): return "sub_core"
    if c.startswith(("don_", "rel_")): return "donation"
    if c.startswith("dist_"): return "dist"
    if c.startswith(("zip_", "geo_")): return "geo"
    if c.startswith("tk13_"): return "tk13"
    if c.startswith("tk_loc_"): return "tk_loc"
    if c.startswith("tk_"): return "tickets"
    if c.startswith("taste_"): return "taste"
    return "other"

xgb = build_pipeline("xgb", "none")
cv_prune = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=7)
base = fold_scores(xgb, X_train, y, cv_prune)
G = {c: group(c) for c in X_train}
rows = []
for g in sorted(set(G.values())):
    s = fold_scores(xgb, X_train[[c for c in X_train if G[c] != g]], y, cv_prune)
    rows.append((g, sum(v == g for v in G.values()), s.mean(), (s - base).mean()))
print("all features:", base.mean().round(4))
pd.DataFrame(rows, columns=["dropped_group", "n_cols", "auroc", "delta"]).sort_values("delta")

all features: 0.9794


,dropped_group,n_cols,auroc,delta
3,id,1,0.966113,-0.013293
5,sub_core,19,0.971386,-0.008020
1,donation,9,0.977818,-0.001588
0,dist,5,0.978261,-0.001145
7,tickets,16,0.978748,-0.000658
4,sub_cat,21,0.978865,-0.000541
8,tk13,3,0.979149,-0.000257
9,tk_loc,5,0.979700,0.000294
6,taste,18,0.981028,0.001622
2,geo,8,0.981218,0.001812


In [25]:
imp = cv_permutation_importance(xgb, X_train, y, RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1))
imp.head(20)

sub_seasons_since_last    0.018904
acct_id_rank              0.018849
sub_subscribed_2013       0.002839
don_avg_gift_log          0.002351
don_years_since_first     0.002209
dist_min                  0.001793
tk_avg_price_level        0.001714
don_n                     0.001592
sub_n_seasons             0.000844
sub_years_since_first     0.000795
don_lifetime_log          0.000745
sub_current_streak        0.000732
sub_frac_last3            0.000663
dist_peninsula            0.000446
taste_era_baroque         0.000414
zip_in_ca                 0.000404
tk13_n_tickets            0.000308
sub_tier_last             0.000181
dist_berkeley             0.000164
dist_walnutcreek          0.000162
dtype: float64

In [26]:
# AUROC vs. number of top-ranked features (XGBoost and logistic regression)
lr_curve = build_pipeline("logreg", "none", C=0.03)
rows = []
for k in [6, 10, 15, 20, 30, 45, X_train.shape[1]]:
    cols = list(imp.index[:k])
    sx = fold_scores(xgb, X_train[cols], y, cv_prune)
    sl = fold_scores(lr_curve, X_train[cols], y, cv_prune)
    rows.append((k, sx.mean(), sx.std(), sl.mean(), sl.std()))
curve = pd.DataFrame(rows, columns=["n_features", "xgb_auroc", "xgb_std", "logreg_auroc", "logreg_std"])
curve["mean_auroc"] = curve[["xgb_auroc", "logreg_auroc"]].mean(axis=1)
curve

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 inst

,n_features,xgb_auroc,xgb_std,logreg_auroc,logreg_std,mean_auroc
0,6,0.982219,0.007594,0.958312,0.016383,0.970265
1,10,0.986789,0.006442,0.976320,0.011115,0.981555
2,15,0.984724,0.007489,0.976623,0.010820,0.980673
3,20,0.985142,0.006552,0.978893,0.009267,0.982017
4,30,0.983902,0.007747,0.977984,0.010625,0.980943
5,45,0.983240,0.008218,0.980274,0.009164,0.981757
6,105,0.979082,0.009307,0.973576,0.007490,0.976329


Choose **K** from the curve: the number of top-ranked features with the highest AUROC averaged over the two evaluators (XGBoost and logistic regression). Averaging two very different models keeps the choice from depending on one model's quirks. Differences between neighbouring sizes are small relative to the fold-to-fold std, so the size is a mild preference, not a sharp optimum.

In [27]:
K = int(curve.loc[curve["mean_auroc"].idxmax(), "n_features"])
FEATURES = list(imp.index[:K])
FEATURES_NO_ID = [c for c in FEATURES if c != "acct_id_rank"]   # for the no-ID fallback
print("K =", K, FEATURES)
Xs = X_train[FEATURES]

K = 20 ['sub_seasons_since_last', 'acct_id_rank', 'sub_subscribed_2013', 'don_avg_gift_log', 'don_years_since_first', 'dist_min', 'tk_avg_price_level', 'don_n', 'sub_n_seasons', 'sub_years_since_first', 'don_lifetime_log', 'sub_current_streak', 'sub_frac_last3', 'dist_peninsula', 'taste_era_baroque', 'zip_in_ca', 'tk13_n_tickets', 'sub_tier_last', 'dist_berkeley', 'dist_walnutcreek']


### 6. Hyperparameter tuning (selected features)
Each member is tuned on repeated stratified 5-fold CV (2 repeats, seed 3), scoring AUROC. Logistic regression: grid over `C` and class weighting. Random forest: grid over leaf size, `max_features` and class weighting (300 trees while searching, 500 in the final model). XGBoost and HistGradientBoosting: random search of 24 configurations each over number of trees, depth, learning rate, regularisation and class weighting (XGBoost also over `min_child_weight` and row/column subsampling; HistGradientBoosting over `min_samples_leaf`).

In [28]:
cv_tune = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=3)

lr_res = grid_search("logreg", {"C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3], "imbalance": ["none", "weight"]},
                     Xs, y, cv_tune)
lr_res.head(8)

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 inst

,imbalance,C,auroc_mean,auroc_std
0,none,3.00,0.984316,0.006485
1,none,1.00,0.984085,0.006618
2,none,0.30,0.983631,0.006995
3,weight,0.03,0.983525,0.006588
4,weight,0.10,0.983477,0.006224
5,weight,0.30,0.983294,0.006087
6,weight,3.00,0.983271,0.006010
7,weight,1.00,0.983256,0.006040


In [29]:
rf_res = grid_search("rf", {"min_samples_leaf": [3, 8, 20], "max_features": ["sqrt", 0.5],
                           "imbalance": ["none", "weight"]}, Xs, y, cv_tune, n_estimators=300)
rf_res

,imbalance,min_samples_leaf,max_features,auroc_mean,auroc_std
0,weight,8,sqrt,0.985548,0.006125
1,none,20,sqrt,0.985344,0.006591
2,weight,20,sqrt,0.985286,0.006384
3,weight,3,sqrt,0.985143,0.006019
4,none,3,sqrt,0.985110,0.006347
5,none,8,sqrt,0.985089,0.006538
6,none,8,0.5,0.984574,0.006440
7,none,20,0.5,0.984380,0.006591
8,weight,20,0.5,0.984063,0.006447
9,weight,8,0.5,0.983660,0.006277


In [30]:
xgb_res = xgb_random_search(Xs, y, cv_tune, n_iter=24, seed=0)
xgb_res.head(10)

,config,imbalance,params,auroc_mean,auroc_std
0,20,none,"{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weig...",0.985093,0.005844
1,15,none,"{'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.05, 'min_child_weig...",0.985024,0.006514
2,11,none,"{'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.1, 'min_child_weigh...",0.984768,0.007737
3,22,weight,"{'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.02, 'min_child_weig...",0.984624,0.006014
4,3,none,"{'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weig...",0.984573,0.006887
5,0,weight,"{'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.02, 'min_child_weig...",0.983759,0.006406
6,10,none,"{'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'min_child_weigh...",0.983600,0.006924
7,14,weight,"{'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weig...",0.983444,0.007590
8,16,weight,"{'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.1, 'min_child_weigh...",0.983020,0.008168
9,17,none,"{'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'min_child_weigh...",0.982858,0.007585


In [31]:
hgb_res = hgb_random_search(Xs, y, cv_tune, n_iter=24, seed=0)
hgb_res.head(10)

,config,imbalance,params,auroc_mean,auroc_std
0,12,weight,"{'max_depth': 2, 'learning_rate': 0.03, 'max_iter': 300, 'min_samples_leaf':...",0.985086,0.006060
1,16,none,"{'max_depth': 2, 'learning_rate': 0.1, 'max_iter': 150, 'min_samples_leaf': ...",0.984952,0.005950
2,6,weight,"{'max_depth': 2, 'learning_rate': 0.05, 'max_iter': 150, 'min_samples_leaf':...",0.984645,0.006911
3,15,weight,"{'max_depth': 2, 'learning_rate': 0.1, 'max_iter': 150, 'min_samples_leaf': ...",0.984513,0.006647
4,0,weight,"{'max_depth': 2, 'learning_rate': 0.1, 'max_iter': 300, 'min_samples_leaf': ...",0.984231,0.006810
5,13,none,"{'max_depth': 4, 'learning_rate': 0.1, 'max_iter': 500, 'min_samples_leaf': ...",0.984085,0.006913
6,8,weight,"{'max_depth': 2, 'learning_rate': 0.1, 'max_iter': 500, 'min_samples_leaf': ...",0.983872,0.007003
7,5,weight,"{'max_depth': 2, 'learning_rate': 0.1, 'max_iter': 300, 'min_samples_leaf': ...",0.983851,0.006794
8,17,weight,"{'max_depth': 4, 'learning_rate': 0.1, 'max_iter': 150, 'min_samples_leaf': ...",0.983838,0.006841
9,20,weight,"{'max_depth': 3, 'learning_rate': 0.05, 'max_iter': 150, 'min_samples_leaf':...",0.983730,0.007891


In [32]:
lr_b, rf_b, xgb_b, hgb_b = lr_res.iloc[0], rf_res.iloc[0], xgb_res.iloc[0], hgb_res.iloc[0]
MEMBERS = {
    "logreg": {"imbalance": lr_b["imbalance"], "params": {"C": float(lr_b["C"])}},
    "rf":     {"imbalance": rf_b["imbalance"],
               "params": {"n_estimators": 500, "min_samples_leaf": int(rf_b["min_samples_leaf"]),
                          "max_features": rf_b["max_features"]}},
    "xgb":    {"imbalance": xgb_b["imbalance"], "params": xgb_b["params"]},
    "hgb":    {"imbalance": hgb_b["imbalance"], "params": hgb_b["params"]},
}
MEMBERS

{'logreg': {'imbalance': 'none', 'params': {'C': 3.0}},
 'rf': {'imbalance': 'weight',
  'params': {'n_estimators': 500,
   'min_samples_leaf': 8,
   'max_features': 'sqrt'}},
 'xgb': {'imbalance': 'none',
  'params': {'n_estimators': 300,
   'max_depth': 4,
   'learning_rate': 0.03,
   'min_child_weight': 1.0,
   'reg_lambda': 5.0,
   'subsample': 0.8,
   'colsample_bytree': 1.0}},
 'hgb': {'imbalance': 'weight',
  'params': {'max_depth': 2,
   'learning_rate': 0.03,
   'max_iter': 300,
   'min_samples_leaf': 20,
   'l2_regularization': 1.0}}}

### 7. Ensemble vs. its members
Equal-weight average of percentile ranks (AUROC only depends on ordering). Scored on a fresh CV seed (21) that played no part in selection or tuning: each member alone, every pair, the three-model ensemble without HistGradientBoosting, the final four-model ensemble, and the final ensemble without the account-ID feature.

In [33]:
cv_final = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=21)
parts = {n: build_pipeline(n, m["imbalance"], **m["params"]) for n, m in MEMBERS.items()}
cands = dict(parts)
for a, b in itertools.combinations(parts, 2):
    cands[f"{a} + {b}"] = RankAverager([(a, parts[a]), (b, parts[b])])
cands["logreg + rf + xgb"] = build_ensemble({k: MEMBERS[k] for k in ["logreg", "rf", "xgb"]})
cands["logreg + rf + xgb + hgb (final)"] = build_ensemble(MEMBERS)
compare(cands, Xs, y, cv_final)

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 inst

,auroc_mean,auroc_std,n_folds
logreg + rf + xgb,0.987716,0.004024,15.0
logreg + rf + xgb + hgb (final),0.987617,0.003787,15.0
logreg + rf,0.987576,0.004295,15.0
logreg + xgb,0.987424,0.003821,15.0
logreg + hgb,0.986996,0.003404,15.0
rf + hgb,0.986728,0.004159,15.0
xgb + hgb,0.986349,0.003994,15.0
rf + xgb,0.986332,0.004629,15.0
xgb,0.985888,0.004592,15.0
hgb,0.985492,0.003893,15.0


In [34]:
compare({"final ensemble, no acct_id_rank": build_ensemble(MEMBERS)}, X_train[FEATURES_NO_ID], y, cv_final)

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 inst

,auroc_mean,auroc_std,n_folds
"final ensemble, no acct_id_rank",0.977174,0.008046,15.0


### 8. Final model and submission
Saves the chosen features and hyperparameters to `results/final_config.json` (so `scripts/make_submission.py` can rebuild the model), then fits on all training data and writes `submission.csv` (columns `ID`, `Predicted`; scores are percentile ranks, which is all AUROC needs).

In [35]:
save_config(FEATURES, MEMBERS)

sub = write_submission(build_ensemble(MEMBERS), Xs, y, X_test[FEATURES], path=SUBS / "submission.csv")

# format checks against the sample submission
sample = pd.read_csv(DATA / "sample_submission.csv")
assert (sub["ID"].values == sample["ID"].values).all(), "IDs/order differ from sample_submission"
assert sub["Predicted"].notna().all() and sub["Predicted"].between(0, 1).all()
print("submission.csv written:", sub.shape)
sub.head()

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


submission.csv written: (2975, 2)


,ID,Predicted
0,001i000000NuQ6Y,0.894790
1,001i000000NuQXz,0.940336
2,001i000000NuRDC,0.900840
3,001i000000NuQeg,0.761387
4,001i000000NuOQc,0.921597


### Optional fallback: same model without the account-ID feature

In [36]:
sub_noid = write_submission(build_ensemble(MEMBERS), X_train[FEATURES_NO_ID], y, X_test[FEATURES_NO_ID],
                            path=SUBS / "submission_no_id.csv")
print("submission_no_id.csv written:", sub_noid.shape)

c:\Users\colby\aipi\modeling_process\kaggle_competition\classical-music-kaggle-competition\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


submission_no_id.csv written: (2975, 2)
